Crameri2012Case1_Relaxation
======

This notebook reproduce the case 1 (a cosine perturbation of the surface) with true free surface in Crameri *et al.* (2012). 

**Keywords:** Sticky air, Free surface

**References**
1. Crameri, F., Schmeling, H., Golabek, G. J., Duretz, T., Orendt, R., Buiter, S. J. H., ... & Tackley, P. J. (2012). A comparison of numerical surface topography calculations in geodynamic modelling: an evaluation of the ‘sticky air’method. Geophysical Journal International, 189(1), 38-54.
   
|  |  |
| :---: | :---: |
| <img src="./images/crameri2012case1.png" width="100%"> | <img src="./images/crameri2012case1_ALEIB_topo.png" width="45%"> |

In [ ]:
from underworld import UWGeodynamics as GEO
from underworld import visualisation as vis 

import underworld.function as fn
import math
import numpy as np
import os

In [ ]:
u = GEO.UnitRegistry

KL = 700 * u.kilometer
K_viscosity = 1e21  * u.pascal * u.second
K_density   = 3300 * u.kilogram / u.meter**3

KM = K_density * KL**3
Kt = KM/ ( KL * K_viscosity )

GEO.scaling_coefficients["[length]"] = KL
GEO.scaling_coefficients["[time]"] = Kt
GEO.scaling_coefficients["[mass]"]= KM

In [ ]:
longtest = False
model_end_time = 10 * u.kiloyear
elementRes = (128,80)

if "UW_LONGTEST" in os.environ or longtest:
    model_end_time = 100 * u.kiloyear
    elementRes = (256,80)

Model = GEO.Model(elementRes=elementRes,
                  minCoord=(0. * u.kilometer, 0. * u.kilometer),  
                  maxCoord=(2800. * u.kilometer, 875. * u.kilometer),
                  gravity=(0.0, -10 * u.meter / u.second**2))

dt = 1.*u.kiloyear
dt_str = "%.1f" %(dt.m)
checkpoint_interval = 1.*u.kiloyear
fdir = "1_23_07_FreeSurfaceALEIB_Crameri2012CaseCase1_Relaxation_dt"+dt_str+"ka"
Model.outputDir = fdir

In [ ]:
wavelength = GEO.nd(Model.maxCoord[0])
amplitude  = GEO.nd(7*u.kilometer)
offset     = GEO.nd(700.*u.kilometer)
k = 2. * math.pi / wavelength

coord = fn.coord()
perturbationFn = offset + amplitude*fn.math.cos(k*coord[0])

zinit = 700*u.kilometer
Model.inter_wall = Model._get_InternalwallSets(zinit)

with Model.mesh.deform_mesh():
     Model.mesh.data[Model.inter_wall.data, 1] = perturbationFn.evaluate(Model.inter_wall)[:,0]

Model._freeSurface_ALEIB = True 
Model.freeSurface = True 
Model._freeSurface.solve(0.)

In [ ]:
# recreate swarm as mesh deformed
import underworld as uw
from collections import OrderedDict 

Model.swarm_variables = OrderedDict()
Model.swarm = uw.swarm.Swarm(mesh=Model.mesh, particleEscape=True)
Model.swarm.allow_parallel_nn = True
GEO.rcParams['swarm.particles.per.cell.2D']=36
particlesPerCell = GEO.rcParams["swarm.particles.per.cell.2D"]
Model._swarmLayout = uw.swarm.layouts.PerCellSpaceFillerLayout(swarm=Model.swarm,particlesPerCell=particlesPerCell)
Model.swarm.populate_using_layout(layout=Model._swarmLayout)
Model._initialize()

In [ ]:
air_Shape = coord[1] >= perturbationFn
li_Shape = (coord[1] > GEO.nd(600*u.kilometer)) & (coord[1] <= perturbationFn)
ma_Shape = coord[1] <= GEO.nd(600*u.kilometer)

air = Model.add_material(name="Air", shape=air_Shape)
li  = Model.add_material(name="Lithosphere", shape=li_Shape)
ma  = Model.add_material(name="Mantle Asthenosphere", shape=ma_Shape)

In [ ]:
Fig = vis.Figure(figsize=(1200,400))
Fig.Mesh(Model.mesh)
Fig.save("Fig_CrameriCase1_mesh0.png")
Fig.show()

In [ ]:
npoints = 5000
coords = np.ndarray((npoints, 2))
coords[:, 0] = np.linspace(GEO.nd(Model.minCoord[0]), GEO.nd(Model.maxCoord[0]), npoints)
coords[:, 1] = offset + amplitude*np.cos(k*coords[:, 0])
surf_tracers = Model.add_passive_tracers(name="Surface",vertices=coords)

In [ ]:
Fig = vis.Figure(figsize=(1200,400))
#Fig.Points(surf_tracers, pointSize=4.0)
Fig.Points(Model.swarm, Model.materialField,fn_size=2.0,discrete=True,colourBar=False)
Fig.Mesh(Model.mesh)
Fig.save("Fig_CrameriCase1_0.png")
Fig.show()

In [ ]:
air.density = 0. * u.kilogram / u.meter**3
li.density = 3300. * u.kilogram / u.metre**3 
ma.density = 3300. * u.kilogram / u.metre**3

air.viscosity = 1e18 * u.pascal * u.second
li.viscosity  =  1e23 * u.pascal * u.second                             
ma.viscosity  =  1e21 * u.pascal * u.second

In [ ]:
Model.set_velocityBCs(left=[0., None], right=[0., None], top=[None, None], bottom=[0.,0.])
#Model.set_velocityBCs(left=[0., None], right=[0., None], top=[None, 0.], bottom=[0.,0.]) 

In [ ]:
Model.solver.set_inner_method("mumps")

In [ ]:
Model.run_for(model_end_time, checkpoint_interval= checkpoint_interval, dt= dt)

In [ ]:
Fig.save("Fig_CrameriCase1_1.png")
Fig.show()